# Chapter 18 — Connecting Everything with Graphs

**Source:** Wengrow, *A Common-Sense Guide to Data Structures and Algorithms*, Ch. 18

## Why graphs?
Store friendships as a list of pairs: `[["Alice","Bob"], ["Bob","Cynthia"], ...]`.
"Who are Alice's friends?" → you must scan **every pair** → O(N).
A **graph** answers it in **O(1)**.

A graph is the data structure for **relationships**: nodes connected by lines.

## Graphs vs trees
**Every tree is a graph, but not every graph is a tree.** A tree is a graph that
1. has **no cycles** (A–B–C–A would be a cycle), and
2. is **connected** (every node is reachable from every other).

Graphs may have cycles and disconnected parts (e.g. a new user with no friends yet).

## Vocabulary
| Term | Meaning |
|---|---|
| **Vertex** (plural vertices) | a node |
| **Edge** | a line between two vertices |
| **Adjacent / neighbours** | two vertices joined by an edge |
| **Connected graph** | every vertex is reachable from every other |
| **Path** | a specific sequence of edges from one vertex to another |
| **Directed graph** | edges have a direction (Alice *follows* Bob, but not back) |
| **Weighted graph** | edges carry a number (distance, price, time …) |

## 1. Representing a graph

### a) Bare-bones: a dict of lists (adjacency list)
Key = vertex, value = list of neighbours. Looking up someone's friends = **one dict lookup = O(1)**.
Undirected: each friendship appears in **both** lists. Directed: only in the follower's list.

### b) Object-oriented: a `Vertex` class
Each vertex stores its `value` and a list of `adjacent_vertices`.
For an **undirected** graph, `add_adjacent_vertex` also adds the reverse link. The early `return` stops infinite ping-pong (Alice adds Bob → Bob adds Alice → Alice already has Bob → stop).

We assume **connected** graphs, so from any single vertex you can reach all the others. For a disconnected graph you'd keep a separate list of all vertices (often a `Graph` class).

### Adjacency list vs adjacency matrix
- **List** (what we use): each vertex keeps a list of its neighbours.
- **Matrix**: a V×V 2-D array where `M[i][j] = 1` (or the weight) if there's an edge i→j. Checking "is there an edge?" is O(1), but it uses V² memory. It's the form you'd build in **NumPy**, and the one you meet in OR/linear algebra (adjacency/cost matrices).

In [1]:
# a) bare-bones adjacency list with a dict
friends = {
    "Alice":   ["Bob", "Diana", "Fred"],
    "Bob":     ["Alice", "Cynthia", "Diana"],
    "Cynthia": ["Bob"],
    "Diana":   ["Alice", "Bob", "Fred"],
    "Elise":   ["Fred"],
    "Fred":    ["Alice", "Diana", "Elise"],
}
print(friends["Alice"])                 # O(1)

followees = {                           # directed
    "Alice":   ["Bob", "Cynthia"],
    "Bob":     ["Cynthia"],
    "Cynthia": ["Bob"],
}


# b) object-oriented, undirected
class Vertex:
    def __init__(self, value):
        self.value = value
        self.adjacent_vertices = []

    def add_adjacent_vertex(self, vertex):
        if vertex in self.adjacent_vertices:     # already linked -> stop the ping-pong
            return
        self.adjacent_vertices.append(vertex)
        vertex.add_adjacent_vertex(self)         # mutual friendship

    def __repr__(self):
        return f"Vertex({self.value!r})"


def build_undirected(edges):
    """Helper: build vertices from a list of (a, b) pairs. Returns {name: Vertex}."""
    v = {}
    for a, b in edges:
        for name in (a, b):
            if name not in v:
                v[name] = Vertex(name)
        v[a].add_adjacent_vertex(v[b])
    return v

['Bob', 'Diana', 'Fred']


## 2. Graph search

"Search" in graphs usually means: **starting from one vertex, find (a path to) another vertex.** Uses:
- find any vertex when you only hold one vertex
- check whether two vertices are **connected** at all
- **traverse**: visit every vertex to do something with it

### The key difference from trees: cycles
Mohammed → Felicia → Zeina → Mohammed → … would loop forever.
So every graph search keeps a record of **visited vertices**. The book uses a hash table; in Python a **`set`** is the natural fit (O(1) membership).

The example social network used in the walk-throughs:

                      Alice
           /       /        \         \
        Bob     Candy       Derek ---- Elaine
         |        |           |
        Fred      |          Gina
         |        |           |
        Helen ----+          Irena

(edges: Alice–Bob, Alice–Candy, Alice–Derek, Alice–Elaine, Derek–Elaine,
 Bob–Fred, Fred–Helen, Helen–Candy, Derek–Gina, Gina–Irena)

In [ ]:
net = build_undirected([
    ("Alice", "Bob"), ("Alice", "Candy"), ("Alice", "Derek"), ("Alice", "Elaine"),
    ("Bob", "Fred"), ("Fred", "Helen"), ("Candy", "Helen"),
    ("Derek", "Elaine"), ("Derek", "Gina"), ("Gina", "Irena"),
])
print(net["Alice"].adjacent_vertices)

## 3. Depth-first search (DFS): recursion

Same idea as BST traversal (Ch. 15) and filesystem traversal (Ch. 10), plus a visited set:
1. Mark the current vertex as visited.
2. Loop over its neighbours.
3. Already visited → skip it. Not visited → **recursively DFS** into it.

DFS goes **as deep/far as possible** first and only comes back when it hits a dead end (through the call stack).
Order from Alice: Alice → Bob → Fred → Helen → Candy → (back up) → Derek → Elaine → Gina → Irena.

For a *search* (not a full traversal), return the vertex as soon as its value matches and pass the result back up the recursion.

⚠️ **Same Python trap again:** the book's `visited_vertices={}` default argument would be shared between separate calls (Ch. 12, Ch. 17). Use `None`.

In [ ]:
def dfs_traverse(vertex, visited=None):
    if visited is None:
        visited = set()
    visited.add(vertex.value)
    print(vertex.value, end=" ")
    for adj in vertex.adjacent_vertices:
        if adj.value in visited:
            continue
        dfs_traverse(adj, visited)


def dfs(vertex, search_value, visited=None):
    """Return the vertex with value == search_value, or None."""
    if visited is None:
        visited = set()
    if vertex.value == search_value:
        return vertex
    visited.add(vertex.value)
    for adj in vertex.adjacent_vertices:
        if adj.value in visited:
            continue
        found = dfs(adj, search_value, visited)
        if found:
            return found                     # pass the result back up the call stack
    return None


dfs_traverse(net["Alice"]); print()
print(dfs(net["Alice"], "Irena"))
print(dfs(net["Alice"], "Zed"))

## 4. Breadth-first search (BFS): a queue, no recursion

1. Mark the start vertex as visited and put it in a **queue**.
2. While the queue isn't empty: **dequeue** the front vertex (= current).
3. For each neighbour that isn't visited yet: mark it visited and **enqueue** it.

BFS visits all **direct neighbours first**, then their neighbours, and so on, spreading out in rings.
Order from Alice: Alice → Bob, Candy, Derek, Elaine (all 1st-degree) → Fred, Helen, Gina (2nd-degree) → Irena.

This is the queue from Ch. 9/14 in action. In Python use `collections.deque` (O(1) `popleft`).
Detail: mark a vertex as visited **when you enqueue it**, not when you dequeue it. Otherwise the same vertex can be queued twice.

## 5. DFS vs BFS: which one?
| Want to… | Use |
|---|---|
| **stay close** to the start (e.g. Alice's direct friends, fewest hops) | **BFS** |
| **get far away quickly** (e.g. a great-grandchild in a family tree) | **DFS** |

In [ ]:
from collections import deque

def bfs_traverse(start):
    queue = deque([start])
    visited = {start.value}
    while queue:
        current = queue.popleft()                 # dequeue the front
        print(current.value, end=" ")
        for adj in current.adjacent_vertices:
            if adj.value not in visited:
                visited.add(adj.value)            # mark when ENQUEUING
                queue.append(adj)

bfs_traverse(net["Alice"])

## 6. Efficiency of graph search: O(V + E)

Worst case, both DFS and BFS visit **every vertex** *and* check **every neighbour** of each vertex.
So the number of vertices alone isn't enough. Two graphs with 5 vertices:
- fully linked (A–B–C–D–E, 8 edges): 5 visits + 16 neighbour checks = **21 steps**
- a star (V in the middle, 4 edges): 5 visits + 8 neighbour checks = **13 steps**

**V** = number of vertices, **E** = number of edges.
In an undirected graph each edge is checked **twice** (once from each end), so the real count is V + 2E. Big O drops the 2 → **O(V + E)**.

More edges → more work, even with the same number of vertices.
(Side note from the book: graph **databases** like Neo4j are built for this kind of data.)

## 7. Weighted graphs

Each edge carries a **weight**: miles, price, time …
Weighted graphs can also be directed: Dallas → Toronto $138, Toronto → Dallas $216.

In code, `adjacent_vertices` becomes a **dict**: `{neighbour_vertex: weight}`.

## 8. The shortest path problem & Dijkstra's algorithm (1959)

The flight graph (directed, weights = prices):

    Atlanta  → Boston  $100      Boston → Chicago $120
    Atlanta  → Denver  $160      Boston → Denver  $180
    Denver   → Chicago $40       Chicago → El Paso $80
    Denver   → El Paso $140      El Paso → Boston $100

Question: **cheapest way from Atlanta to El Paso**, any number of stops?
(Atlanta–Denver–El Paso = $300, but Atlanta–Denver–Chicago–El Paso = **$280**.)

**Free bonus:** Dijkstra gives the cheapest price from the start to **every** reachable city, not just the destination.

### Two tables (dicts)
- `cheapest_prices`: city → cheapest known total price **from the start**. Start city = 0.
- `cheapest_previous`: city → the **stop just before it** on that cheapest route. Used at the end to rebuild the path.

### The steps
1. Current city = start.
2. For each neighbour: price via current = `cheapest_prices[current] + edge price`.
3. If that's cheaper than what's known (or the city is new) → **update both tables**.
4. Next current city = the **unvisited** city with the **lowest** `cheapest_prices` value.
5. Repeat until every known city is visited.
6. **Rebuild the path backwards** from the destination with `cheapest_previous`:
   El Paso ← Chicago ← Denver ← Atlanta → reverse → **Atlanta → Denver → Chicago → El Paso**.

Final tables:

| | Atlanta | Boston | Chicago | Denver | El Paso |
|---|---|---|---|---|---|
| cheapest price | 0 | 100 | 200 | 160 | 280 |
| previous stop | – | Atlanta | Denver | Atlanta | Chicago |

Notice step 3 where Chicago is first $220 (via Boston) and later improves to $200 (via Denver). Values only ever go **down**.

In [ ]:
class City:
    def __init__(self, name):
        self.name = name
        self.routes = {}                  # {City: price}

    def add_route(self, city, price):
        self.routes[city] = price


def dijkstra_shortest_path(starting_city, final_destination):
    cheapest_prices = {starting_city.name: 0}
    cheapest_previous = {}
    unvisited = []                        # known but not yet visited cities
    visited = set()

    current = starting_city
    while current:
        visited.add(current.name)
        unvisited = [c for c in unvisited if c is not current]

        for adj, price in current.routes.items():
            if adj.name not in visited and adj not in unvisited:
                unvisited.append(adj)     # discovered a new city

            price_through_current = cheapest_prices[current.name] + price
            if adj.name not in cheapest_prices or price_through_current < cheapest_prices[adj.name]:
                cheapest_prices[adj.name] = price_through_current
                cheapest_previous[adj.name] = current.name

        # next: the cheapest unvisited city (from the START)
        current = min(unvisited, key=lambda c: cheapest_prices[c.name]) if unvisited else None

    # rebuild the path backwards
    path = []
    name = final_destination.name
    while name != starting_city.name:
        path.append(name)
        name = cheapest_previous[name]
    path.append(starting_city.name)
    return path[::-1], cheapest_prices


atlanta, boston, chicago, denver, el_paso = (City(n) for n in
    ["Atlanta", "Boston", "Chicago", "Denver", "El Paso"])
atlanta.add_route(boston, 100);  atlanta.add_route(denver, 160)
boston.add_route(chicago, 120);  boston.add_route(denver, 180)
chicago.add_route(el_paso, 80)
denver.add_route(chicago, 40);   denver.add_route(el_paso, 140)
el_paso.add_route(boston, 100)

path, prices = dijkstra_shortest_path(atlanta, el_paso)
print(path)       # ['Atlanta', 'Denver', 'Chicago', 'El Paso']
print(prices)     # cheapest price to EVERY city

### Efficiency of Dijkstra
- With a **plain list** for `unvisited` (as above): finding the min is O(V) each time → worst case **O(V²)**.
- With a **priority queue (heap)** from Ch. 16 → faster, about **O((V + E) log V)** with `heapq`. The book leaves this as a suggestion; it's Bonus Exercise 6 below.
- Either way it's far better than trying every possible path.

(One condition the book doesn't spell out: Dijkstra assumes **no negative edge weights**. For negative weights you use **Bellman–Ford**, one of the algorithms the book names at the end.)

## Summary
| Topic | Key point |
|---|---|
| Graph | vertices + edges; trees are graphs with no cycles that are connected |
| Representation | adjacency list (dict of lists / `Vertex` class) or adjacency matrix |
| DFS | recursion + visited set; goes deep first |
| BFS | queue (`deque`) + visited set; goes wide first (nearest first) |
| Search cost | **O(V + E)** for both |
| Weighted graph | edges carry weights; neighbours stored as `{vertex: weight}` |
| Dijkstra | cheapest path from one start to all vertices; two tables; rebuild the path backwards; O(V²) with a list, faster with a heap |

The book's further-reading list for graphs is very OR-relevant: minimum spanning tree, topological sort, Floyd–Warshall, Bellman–Ford, graph colouring. In practice, the Python library **`networkx`** implements all of these.

## Exercises (book solutions: p. 461)

### Exercise 1: recommendation engine
**Input:** an undirected graph where each vertex is a product and an edge means "similar product, recommend it". Edges:

    brush–nail polish, nail polish–nails, nail polish–eye shadow, eye shadow–eye glasses,
    drill–hammer, hammer–nails, hammer–saw, nails–pins, nails–needles, pins–needles,
    saw–knife, knife–fork, knife–spoon

**Question:** a user is browsing **"nails"**. Which products are recommended? (= its direct neighbours only)

### Exercise 2: DFS order (on paper)
**Input:** this undirected graph, starting at **A**. When you have a choice, visit the neighbour that's **earliest in the alphabet** first.

            O
             \
              J --- F
              |     |
              E --- B
                     \
                      A --- C
                     /       \
          L --- H --- D       G --- K
                |     |
                M --- I --- N --- P

(edges: O–J, J–F, J–E, F–B, E–B, B–A, A–C, C–G, G–K, A–D, L–H, H–D, H–M, D–I, M–I, I–N, N–P)

**Task:** write out the order in which DFS visits all 16 vertices.

### Exercise 3: BFS order (on paper)
**Input:** the same graph, start at **A**, alphabetical tie-breaking.
**Task:** write out the BFS visiting order.

### Exercise 4: BFS *search*
**Input:** a start `Vertex` and a `search_value` (e.g. `net["Alice"]`, `"Irena"`).
**Task:** modify `bfs_traverse` into `bfs(start, search_value)` that **returns the vertex's value** if found, else `None` (as `dfs` does for DFS).

### Exercise 5: shortest path in an UNweighted graph
**Input:** two vertices of an undirected social network. "Shortest" = the **fewest hops**. The network is a ring:

    Idris – Kamil – Lina – Sasha – Marco – Ken – Talia – Idris

Idris → Lina is **2 hops** via Kamil (or 5 hops the other way round).
**Task:** write `shortest_path(start, end)` that returns the path as a list, e.g. `["Idris", "Kamil", "Lina"]`.
*Book hint:* combine **BFS** (it finds the nearest vertices first) with Dijkstra's **"previous stop" table** (to rebuild the path).

### Bonus Exercise 6 (the book's suggestion): Dijkstra with a heap
**Input:** the same `City` flight graph.
**Task:** replace the `unvisited` list + `min()` with **`heapq`** (Ch. 16). Push `(price_so_far, city_name, city)` tuples, pop the cheapest. Hint: a city can end up in the heap more than once, so skip it when you pop a city that's already visited.
Expected: same result, `['Atlanta', 'Denver', 'Chicago', 'El Paso']` and $280.

In [ ]:
# Exercise 1
products = build_undirected([
    ("brush", "nail polish"), ("nail polish", "nails"), ("nail polish", "eye shadow"),
    ("eye shadow", "eye glasses"), ("drill", "hammer"), ("hammer", "nails"), ("hammer", "saw"),
    ("nails", "pins"), ("nails", "needles"), ("pins", "needles"),
    ("saw", "knife"), ("knife", "fork"), ("knife", "spoon"),
])
# answer on paper first, then:
# print([v.value for v in products["nails"].adjacent_vertices])


# Exercises 2 & 3 — same graph, neighbours sorted alphabetically
letters = build_undirected([
    ("O", "J"), ("J", "F"), ("J", "E"), ("F", "B"), ("E", "B"), ("B", "A"), ("A", "C"),
    ("C", "G"), ("G", "K"), ("A", "D"), ("L", "H"), ("H", "D"), ("H", "M"),
    ("D", "I"), ("M", "I"), ("I", "N"), ("N", "P"),
])
for v in letters.values():
    v.adjacent_vertices.sort(key=lambda x: x.value)    # alphabetical tie-breaking
# write your orders first, then check:
# dfs_traverse(letters["A"]); print()
# bfs_traverse(letters["A"])

In [ ]:
# Exercise 4
def bfs(start, search_value):
    pass

# print(bfs(net["Alice"], "Irena"))   # 'Irena'
# print(bfs(net["Alice"], "Zed"))     # None


# Exercise 5
ring = build_undirected([
    ("Idris", "Kamil"), ("Kamil", "Lina"), ("Lina", "Sasha"), ("Sasha", "Marco"),
    ("Marco", "Ken"), ("Ken", "Talia"), ("Talia", "Idris"),
])

def shortest_path(start, end):
    pass

# print(shortest_path(ring["Idris"], ring["Lina"]))   # ['Idris', 'Kamil', 'Lina']


# Bonus Exercise 6
import heapq

def dijkstra_heap(starting_city, final_destination):
    pass

# print(dijkstra_heap(atlanta, el_paso))